In [1]:
from datasets import load_dataset
#os.environ["HF_ENDPOINT"]="https://hf-mirror.com/"
#export HF_ENDPOINT="https://hf-mirror.com"
import os

#os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
#os.environ["HF_HUB_ENABLE_HF_MIRROR"] = "true"

In [12]:
load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1')

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [1]:
from datasets import load_dataset

# for fit lower resouces in Kaggle here
dataset=load_dataset("m-a-p/COIG-CQIA", 'chinese_traditional' , split="train[:500]")
dataset=dataset.train_test_split(test_size=0.1)
print(dataset["train"][0])
dataset

README.md:   0%|          | 0.00/12.7k [00:00<?, ?B/s]

chinese_traditional/chengyu.jsonl:   0%|          | 0.00/82.5k [00:00<?, ?B/s]

chinese_traditional/poem.jsonl:   0%|          | 0.00/30.0k [00:00<?, ?B/s]

(…)raditional/trad-multi-choice-100-2.jsonl:   0%|          | 0.00/58.5k [00:00<?, ?B/s]

(…)_traditional/trad-multi-choice-100.jsonl:   0%|          | 0.00/59.9k [00:00<?, ?B/s]

(…)e_traditional/trad-multi-choice-40.jsonl:   0%|          | 0.00/40.9k [00:00<?, ?B/s]

(…)tional/translate_classical_chinese.jsonl:   0%|          | 0.00/649k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1111 [00:00<?, ? examples/s]

{'instruction': '秋，七月，渤海孝王鸿薨，无子；太后立帝弟蠡吾侯悝为渤海王，以奉鸿祀。诏以定策功，益封梁冀万三千户，封冀弟不疑为颍阳侯，蒙为西平侯，冀子胤为襄邑侯，胡广为安乐侯，赵戒为厨亭侯，袁汤为安国侯。\n帮我把这句话翻译成现代文', 'input': '', 'output': '梁太后封桓帝的弟弟蠡吾侯刘悝为勃海王，以祭祀刘鸿做他的继承人。桓帝下诏，因拥立皇帝决策有功，增封梁冀食邑一万三千户，封梁冀的弟弟梁不疑为颍阳侯，梁蒙为西平侯，梁冀的儿子梁胤为襄邑侯，胡广为安乐侯，赵戒为厨亭侯，袁汤为安国侯。', 'task_type': {'major': ['文本生成'], 'minor': ['文言文翻译']}, 'domain': ['中国传统文化', '文言文'], 'metadata': '暂无元数据信息', 'answer_from': 'human', 'human_verified': True, 'copyright': '暂无版权及作者信息'}


DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'task_type', 'domain', 'metadata', 'answer_from', 'human_verified', 'copyright'],
        num_rows: 450
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'task_type', 'domain', 'metadata', 'answer_from', 'human_verified', 'copyright'],
        num_rows: 50
    })
})

In [9]:
templates="'<|im_start|>user\n{instruction}<|im_end|>','<|im_start|>assistant\n{output}<|im_end|>'"

msg_chatml=[templates.format(instruction=data["instruction"],output=data["output"]) for data in dataset["train"]]

In [3]:
import os
from functools import partial

templates=[
    '<|im_start|>user\n{instruction}<|im_end|>',
    '<|im_start|>assistant\n{output}<|im_end|>'
]

IGNORE_INDEX=-100

def preprocess_func(input, max_length):
    input_ids, attention_mask, labels=[],[],[]
    templates="'<|im_start|>user\n{instruction}<|im_end|>','<|im_start|>assistant\n{output}<|im_end|>'"
    msg_chatml = [templates.format(instruction=data["instruction"],output=data["output"]) for data in dataset["train"]]      
    msg_tokenized=tokenizer(msg_chatml,truncation=False, add_special_tokens=False)
    input_ids+=msg_tokenized["input_ids"]
    attention_mask+=msg_tokenized["attention_mask"]
    labels+=[IGNORE_INDEX]*len(msg_tokenized["input_ids"]) if isHuman else msg_tokenized["input_ids"]

    return {
        "input_ids": input_ids[:max_length],
        "attention_mask": attention_mask[:max_length],
        "labels": labels[:max_length]
    }

dataset_tokenized=dataset.map(
    partial(preprocess_func, max_length=1024), # max sample length 1024 tokens, enough for the dataset
    batched=False,
    num_proc=os.cpu_count(),
    remove_columns=dataset["train"].column_names # do not need this anymore, we have tokens from here on
)

In [ ]:
dataset_tokenized=dataset.map(
    partial(preprocess_func, max_length=1024), # max sample length 1024 tokens, enough for the dataset
    batched=False,
    num_proc=os.cpu_count(),
    remove_columns=dataset["train"].column_names # do not need this anymore, we have tokens from here on
)

In [4]:
!wget https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_linux_aarch64

--2025-01-19 01:43:53--  https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_linux_aarch64
Resolving cdn-media.huggingface.co (cdn-media.huggingface.co)... 3.161.150.28, 3.161.150.103, 3.161.150.49, ...
Connecting to cdn-media.huggingface.co (cdn-media.huggingface.co)|3.161.150.28|:443... connected.
HTTP request sent, awaiting response... 403 Forbidden
2025-01-19 01:43:53 ERROR 403: Forbidden.



In [5]:
from datasets import load_dataset

# for fit lower resouces in Kaggle here
dataset=load_dataset("YeungNLP/moss-003-sft-data", split="train[:500]")
dataset=dataset.train_test_split(test_size=0.1)
print(dataset["train"][0])
dataset

README.md:   0%|          | 0.00/7.33k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


moss-003-sft-data.jsonl:   0%|          | 0.00/8.72G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

{'conversation_id': 221, 'category': 'Brainstorming', 'conversation': [{'human': '如何使用AI技术分析数据？', 'assistant': '使用AI技术分析数据通常需要以下步骤：\n\n1. 收集和准备数据：这是AI分析的第一步。你需要收集数据并将其准备好，以便可以使用AI算法进行分析。这可能包括清洗数据，填充缺失值，将数据转换为数字格式等。\n\n2. 选择合适的AI算法：选择适合你的数据类型和所需结果的AI算法。例如，你可以选择使用回归模型来预测数值，或使用分类模型来将数据分成多个类别。\n\n3. 训练模型：将数据集分成训练集和测试集，并使用训练集训练AI模型。这将使模型能够了解数据的模式和趋势，并为未知数据做出预测。\n\n4. 评估模型：评估模型的性能。这可以通过计算模型的准确性、精确性、召回率等指标来完成。如果模型的性能不理想，则需要调整算法或更改数据。\n\n5. 使用模型进行预测：使用训练好的AI模型对新数据进行预测。这可以帮助你了解数据的趋势和预测未来的趋势。\n\n总之，使用AI技术分析数据需要准备数据、选择算法、训练模型、评估模型和使用模型进行预测。'}, {'human': '你能给我举一个使用AI技术分析数据的实际例子吗？', 'assistant': '当然可以，以下是一个使用AI技术分析数据的实际例子：\n\n假设一个网站想要了解他们的用户在哪些页面上停留时间最长，以及哪些页面容易让用户离开网站。他们可以使用AI技术来分析网站的访问数据，以找到答案。\n\n他们可以按照以下步骤进行：\n\n1. 收集和准备数据：他们可以使用网站分析工具来收集用户访问数据，并将数据准备成可以被AI算法处理的格式。\n\n2. 选择合适的AI算法：他们可以使用聚类算法，将用户访问数据分为不同的组，以发现哪些页面最受欢迎，哪些页面容易让用户离开网站。\n\n3. 训练模型：他们将数据集分为训练集和测试集，并使用训练集训练聚类模型。\n\n4. 评估模型：他们可以评估模型的性能，以确保模型的准确性和可靠性。\n\n5. 使用模型进行预测：最后，他们可以使用训练好的模型，对新的访问数据进行聚类分析，以发现哪些页面是最受欢迎的，哪些页面容易让用户离开网站。\n\n通过这个例子，我们

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'category', 'conversation'],
        num_rows: 450
    })
    test: Dataset({
        features: ['conversation_id', 'category', 'conversation'],
        num_rows: 50
    })
})

In [ ]:
import gc

del tokenizer, lora_model, prepared_model, model

gc.collect()
torch.cuda.empty_cache()

一般开源大模型支持以 alpaca 或 shareGPT 格式的数据集

In [ ]:
# alpaca 格式的数据集应遵循以下格式：
[
  {
    "instruction": "user instruction (required)",
    "input": "user input (optional)",
    "output": "model response (required)",
    "system": "system prompt (optional)",
    "history": [
      ["user instruction in the first round (optional)", "model response in the first round (optional)"],
      ["user instruction in the second round (optional)", "model response in the second round (optional)"]
    ]
  }
]

# 对于 alpaca 格式的数据集，其 dataset_info.json 文件中的列应为：
"dataset_name": {
  "file_name": "dataset_name.json",
  "columns": {
    "prompt": "instruction",
    "query": "input",
    "response": "output",
    "system": "system",
    "history": "history"
  }
}

In [ ]:
# sharegpt 格式的数据集应遵循以下格式

[
  {
    "conversations": [
      {
        "from": "human",
        "value": "user instruction"
      },
      {
        "from": "gpt",
        "value": "model response"
      }
    ],
    "system": "system prompt (optional)",
    "tools": "tool description (optional)"
  }
]

# 对于 sharegpt 格式的数据集，dataset_info.json 文件中的列应该包括：

"dataset_name": {
    "file_name": "dataset_name.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "conversations",
      "system": "system",
      "tools": "tools"
    },
    "tags": {
      "role_tag": "from",
      "content_tag": "value",
      "user_tag": "user",
      "assistant_tag": "assistant"
    }
  }

# 微调数据集

## 英文

‌yahma/alpaca-cleaned‌是一个数据集，由Stanford Alpaca项目团队发布，主要用于语言模型微调。该数据集是LLaMA模型生成的数据经过过滤和清理后的版本，包含约52K条数据。数据集的目的是提供高质量的教学数据，以便更有效地微调语言模型，使其能够更好地遵循指令‌

In [ ]:
load_dataset('yahma/alpaca-cleaned', cache_dir='/kaggle/working/',trust_remote_code=True)

## 中文

* 数据集名称：BelleGroup/train_3.5M_CN
* 数据集标签：【微调数据】【中文】【通用领域】【多轮QA】
* 数据集介绍：BelleGroup/train_3.5M_CN包含约350万条由BELLE项目生成的中文指令数据。数据内容以对话形式给出，包括多轮和单轮对话的数据。采用self-instruct技术。使用生成模型是 text-davinci-003。针对该数据集，新增了指令类别字段，共包括13个类别，有生成，信息抽取，角色扮演，开放式问答，分类，头脑风暴，数学，翻译，代码，摘要生成，创意重写，无害数据，封闭式问答。

In [ ]:
load_dataset('BelleGroup/train_3.5M_CN', cache_dir='/kaggle/working/',trust_remote_code=True)
#eval_dataset = load_dataset('gem/viggo', split='validation',trust_remote_code=True)
#test_dataset = load_dataset('gem/viggo', split='test',trust_remote_code=True)

* 数据集名称：BelleGroup/train_**M_CN
* 数据集标签：【微调数据】【中文】【通用领域】【QA】
* 数据集介绍：50/100/200万条中文ChatGPT指令Belle数据集。包含数个由BELLE项目产生的不同指令类型、不同领域的子集。train_**M_CN采用Self-instruct技术，基于175中文种子任务和text-davinci-003模型生成的。

In [3]:
train_dataset = load_dataset('BelleGroup/train_0.5M_CN', cache_dir='/kaggle/working/',trust_remote_code=True)
#eval_dataset = load_dataset('gem/viggo', split='validation',trust_remote_code=True)
#test_dataset = load_dataset('gem/viggo', split='test',trust_remote_code=True)
#train_dataset = load_dataset('BelleGroup/generated_chat_0.4M', split='train',trust_remote_code=True)

README.md:   0%|          | 0.00/940 [00:00<?, ?B/s]

Belle_open_source_0.5M.json:   0%|          | 0.00/286M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/519255 [00:00<?, ? examples/s]

* 数据集名称：YeungNLP/moss-003-sft-data
* 数据集标签：【微调数据】【中文】【通用领域】【多轮QA】
* 数据集介绍：moss-003-sft-data数据集可用于中文多轮对话指令微调，包含110万中英文多轮对话数据。该数据集来自MOSS项目 中的moss-003-sft-data数据集。在原数据集的基础上，我们去除了冗余信息，仅提取出有效的对话信息，并且调整数据格式，以便在训练中更加灵活地组织数据格式。更多详细信息，可参考MOSS项目介绍。

In [ ]:
#train_dataset = load_dataset('YeungNLP/moss-003-sft-data',, split='train', trust_remote_code=True)
dataset = load_dataset('YeungNLP/moss-003-sft-data', cache_dir='/kaggle/working/',trust_remote_code=True)
dataset

* 数据集名称：fnlp/moss-002-sft-data
* 数据集标签：【微调数据】【中文】【通用领域】【QA】
* 数据集介绍：moss-002-sft-data是由116万条中文ChatGPT多轮对话数据集(约 57 万条英文对话和 59 万条中文对话)。用户提示基于一小组人类编写的种子提示进行了扩展，类似于Self-Instruct的方式。覆盖有用性、忠实性、无害性三个层面。使用 text-davinci-003 生成的。用于训练 MOSS-002 的开源对话数据集。

In [9]:
load_dataset('fnlp/moss-002-sft-data', cache_dir='/kaggle/working/',trust_remote_code=True)
#eval_dataset = load_dataset('gem/viggo', split='validation',trust_remote_code=True)
#test_dataset = load_dataset('gem/viggo', split='test',trust_remote_code=True)

DatasetDict({
    train: Dataset({
        features: ['id', 'prefix', 'num_turns', 'plain_text'],
        num_rows: 1161137
    })
})

In [37]:
import torch
dataset=load_dataset('fnlp/moss-002-sft-data', cache_dir='/kaggle/working/',trust_remote_code=True)
ds = dataset['train'].select(torch.randint(100000, 910000, (10, 1)))
ds['plain_text']

['[Human]: Find five examples of HTML tags and explain what they do.<eoh> [MOSS]: Sure! Here are five examples of HTML tags, along with their respective functions: \n\n1. `<p>`: This tag is used to create a paragraph.\n2. `<div>`: This tag defines a division or section of a web page.\n3. `<span>`: This tag is used to group elements and apply styles to them.\n4. `<a>`: This tag is used to create a link to another page or an anchor point within the same page.\n5. `<img>`: This tag is used to insert an image into a web page.<eoa> [Human]: What are the advantages of using HTML?<eoh> [MOSS]: Some of the key advantages of using HTML include:\n\n1. It is a widely-used standard that enables content to be displayed on any browser or device.\n2. It is relatively easy to learn and use.\n3. It enables the separation of content and presentation.\n4. It enables the use of multimedia, such as images, videos, and audio.\n5. It enables the use of interactive elements, such as forms, buttons, and menus.

In [ ]:
ds['plain_text']

* 数据集名称：shibing624/alpaca-zh
* 数据集标签：【微调数据】【中文】【通用领域】【QA】
* 数据集介绍：Alpaca-GPT-4是使用 self-instruct 技术，基于 175 条中文种子任务和 GPT-4 接口生成的 50K 的指令微调数据集。 

In [8]:
load_dataset('shibing624/alpaca-zh', cache_dir='/kaggle/working/',trust_remote_code=True)
#eval_dataset = load_dataset('gem/viggo', split='validation',trust_remote_code=True)
#test_dataset = load_dataset('gem/viggo', split='test',trust_remote_code=True)

README.md:   0%|          | 0.00/1.65k [00:00<?, ?B/s]

alpaca_gpt4_data_zh.json:   0%|          | 0.00/35.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/48818 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 48818
    })
})

* 数据集名称：Chinese-Vicuna/guanaco_belle_merge_v1.0
* 数据集标签：【微调数据】【中文】【通用领域】【QA】
* 数据集介绍：69万条中文指令Guanaco数据集(Belle50万条+Guanaco19万条)。

In [ ]:
load_dataset('Chinese-Vicuna/guanaco_belle_merge_v1.0',cache_dir='/kaggle/working/',trust_remote_code=True)
#eval_dataset = load_dataset('gem/viggo', split='validation',trust_remote_code=True)
#test_dataset = load_dataset('gem/viggo', split='test',trust_remote_code=True)

* 数据集名称：chatgpt-corpus
* 数据集标签：【微调数据】【中文】【通用领域】【QA】
* 数据集介绍：chatgpt 中文语料库。开源了由 ChatGPT3.5 生成的300万自问自答数据，包括多个领域，可用于用于训练大模型。

# 评价数据集

In [ ]:
load_dataset('hellaswag', cache_dir='/kaggle/working/',trust_remote_code=True)

In [ ]:
load_dataset('EleutherAI/lambada_openai', cache_dir='/kaggle/working/',trust_remote_code=True)